In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Labs"
)

xpt_file = DATA_DIR / "GLU_F.xpt"
csv_file = DATA_DIR / "GLU_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched source values.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 GLU_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTSAF2YR",
    "LBXGLU",
    "LBDGLUSI",
    "LBXIN",
    "LBDINSI",
    "PHAFSTHR",
    "PHAFSTMN"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "GLU_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 3581:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 8:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 3,581 rows x 8 columns."
)


# ============================================================
# 4. CHECK FOR KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Do NOT globally replace it until the exact
# variable/count pattern is validated.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Validated against:
#
# 1. Attached GLU_F.xpt
# 2. CDC 2009-2010 GLU_F codebook
#
# Zero is valid in all three variables.
#

EXPECTED_TINY_COUNTS = {

    "WTSAF2YR": 396,

    "PHAFSTHR": 57,

    "PHAFSTMN": 70
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated GLU_F pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 523:

    raise ValueError(
        f"Unexpected total tiny-value count: {tiny_total}"
    )


print(
    "\nPASS: Tiny-value pattern matches "
    "validated GLU_F pattern."
)


# ============================================================
# 6. RESTORE VALIDATED ZERO VALUES
# ============================================================

#
# Restore ONLY exact TINY_VALUE occurrences
# in the three validated fields.
#
# No blanket replacement.
#

print("\n--- ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    tiny_mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        tiny_mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: unexpected tiny-value count."
        )


    df.loc[
        tiny_mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 523:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zero values restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# dataset should contain.
#
# The ONLY numeric differences from source_df are:
#
# WTSAF2YR tiny -> 0
# PHAFSTHR tiny -> 0
# PHAFSTMN tiny -> 0
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 8. IDENTIFY GENUINE DECIMAL VARIABLES
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# Never round genuine laboratory measurements,
# sample weights, Dietary values, or calculated
# decimal variables.
#

print("\n--- GENUINE DECIMAL CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "genuine fractional values"
        )


# ============================================================
# 9. VALIDATE EXPECTED DECIMAL VARIABLES
# ============================================================

#
# These four variables genuinely contain decimals:
#
# WTSAF2YR
#     Fasting subsample weight
#
# LBDGLUSI
#     Fasting glucose mmol/L
#
# LBXIN
#     Insulin uU/mL
#
# LBDINSI
#     Insulin pmol/L
#

DECIMAL_COLUMNS = [
    "WTSAF2YR",
    "LBDGLUSI",
    "LBXIN",
    "LBDINSI"
]


if fractional_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern differs "
        "from validated GLU_F pattern.\n"
        f"Expected: {DECIMAL_COLUMNS}\n"
        f"Actual:   {fractional_columns}"
    )


print(
    "\nPASS: Genuine decimal variables identified:"
)


for col in DECIMAL_COLUMNS:

    print(
        "   ",
        col
    )


print(
    "\nPASS: Genuine decimals will NOT "
    "be rounded or converted to Int64."
)


# ============================================================
# 10. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51641
    or
    df["SEQN"].max()
    != 62158
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 11. FORMAT ONLY PROVEN WHOLE-NUMBER VARIABLES
# ============================================================

#
# These fields contain whole-number values in this
# released GLU_F file:
#
# SEQN
# LBXGLU
# PHAFSTHR
# PHAFSTMN
#
# They can safely be represented without .0.
#
# NO .round() is used.
#

INTEGER_COLUMNS = [
    col
    for col in df.columns
    if col not in DECIMAL_COLUMNS
]


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_number_check = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not np.all(
            whole_number_check
        ):

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: Whole-number fields formatted "
    "without unnecessary .0."
)


print(
    "PASS: Lab decimal variables remain unchanged."
)


# ============================================================
# 12. CDC CHECK - WTSAF2YR
# ============================================================

#
# CDC:
#
# Fasting Subsample 2 Year MEC Weight
#
# Range:
#     0 to 337218.92
#
# Count:
#     3,581
#
# Missing:
#     0
#
# The 396 tiny values are legitimate zero weights.
#

s = df["WTSAF2YR"]


print("\n--- WTSAF2YR ---")


print(
    "Minimum:",
    s.min()
)


print(
    "Maximum:",
    s.max()
)


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    np.isclose(
        s.min(),
        0,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        337218.92,
        rtol=0,
        atol=0
    )
    and
    int(
        s.notna()
        .sum()
    )
    == 3581
    and
    int(
        s.isna()
        .sum()
    )
    == 0
    and
    int(
        s.eq(0)
        .sum()
    )
    == 396
):

    raise ValueError(
        "WTSAF2YR differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: WTSAF2YR matches CDC."
)


print(
    "PASS: Non-zero sample-weight decimals "
    "remain unchanged."
)


# ============================================================
# 13. CDC CHECK - LBXGLU
# ============================================================

#
# Fasting glucose, mg/dL
#
# CDC:
#
# 36 to 375
# Non-missing = 3,386
# Missing     =   195
#

s = df["LBXGLU"]


print("\n--- LBXGLU ---")


print(
    "Minimum:",
    s.min()
)


print(
    "Maximum:",
    s.max()
)


print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    s.min() == 36
    and
    s.max() == 375
    and
    int(
        s.notna()
        .sum()
    )
    == 3386
    and
    int(
        s.isna()
        .sum()
    )
    == 195
):

    raise ValueError(
        "LBXGLU differs from CDC."
    )


print(
    "PASS: LBXGLU matches CDC."
)


# ============================================================
# 14. CDC CHECK - LBDGLUSI
# ============================================================

#
# Fasting glucose, mmol/L
#
# TRUE DECIMAL VARIABLE.
#
# CDC:
#
# 1.998 to 20.816
# Non-missing = 3,386
# Missing     =   195
#

s = df["LBDGLUSI"]


print("\n--- LBDGLUSI ---")


if not (
    np.isclose(
        s.min(),
        1.998,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        20.816,
        rtol=0,
        atol=0
    )
    and
    int(
        s.notna()
        .sum()
    )
    == 3386
    and
    int(
        s.isna()
        .sum()
    )
    == 195
):

    raise ValueError(
        "LBDGLUSI differs from CDC."
    )


print(
    "PASS: LBDGLUSI matches CDC."
)


print(
    "PASS: LBDGLUSI decimals preserved."
)


# ============================================================
# 15. CDC CHECK - LBXIN
# ============================================================

#
# Insulin, uU/mL
#
# TRUE DECIMAL VARIABLE.
#
# CDC:
#
# 0.88 to 320.22
# Non-missing = 3,337
# Missing     =   244
#

s = df["LBXIN"]


print("\n--- LBXIN ---")


if not (
    np.isclose(
        s.min(),
        0.88,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        320.22,
        rtol=0,
        atol=0
    )
    and
    int(
        s.notna()
        .sum()
    )
    == 3337
    and
    int(
        s.isna()
        .sum()
    )
    == 244
):

    raise ValueError(
        "LBXIN differs from CDC."
    )


print(
    "PASS: LBXIN matches CDC."
)


print(
    "PASS: LBXIN decimals preserved."
)


# ============================================================
# 16. CDC CHECK - LBDINSI
# ============================================================

#
# Insulin, pmol/L
#
# TRUE DECIMAL VARIABLE.
#
# CDC:
#
# 5.28 to 1921.32
# Non-missing = 3,337
# Missing     =   244
#

s = df["LBDINSI"]


print("\n--- LBDINSI ---")


if not (
    np.isclose(
        s.min(),
        5.28,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        1921.32,
        rtol=0,
        atol=0
    )
    and
    int(
        s.notna()
        .sum()
    )
    == 3337
    and
    int(
        s.isna()
        .sum()
    )
    == 244
):

    raise ValueError(
        "LBDINSI differs from CDC."
    )


print(
    "PASS: LBDINSI matches CDC."
)


print(
    "PASS: LBDINSI decimals preserved."
)


# ============================================================
# 17. CDC CHECK - FASTING HOURS
# ============================================================

#
# PHAFSTHR:
#
# Range       = 0 to 39
# Non-missing = 3,535
# Missing     = 46
#
# 57 tiny values were restored to zero.
#

s = df["PHAFSTHR"]


print("\n--- PHAFSTHR ---")


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    s.min() == 0
    and
    s.max() == 39
    and
    int(
        s.notna()
        .sum()
    )
    == 3535
    and
    int(
        s.isna()
        .sum()
    )
    == 46
    and
    int(
        s.eq(0)
        .sum()
    )
    == 57
):

    raise ValueError(
        "PHAFSTHR differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: PHAFSTHR matches CDC."
)


# ============================================================
# 18. CDC CHECK - FASTING MINUTES
# ============================================================

#
# PHAFSTMN:
#
# Range       = 0 to 59
# Non-missing = 3,535
# Missing     = 46
#
# 70 tiny values were restored to zero.
#

s = df["PHAFSTMN"]


print("\n--- PHAFSTMN ---")


print(
    "Zero:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    s.min() == 0
    and
    s.max() == 59
    and
    int(
        s.notna()
        .sum()
    )
    == 3535
    and
    int(
        s.isna()
        .sum()
    )
    == 46
    and
    int(
        s.eq(0)
        .sum()
    )
    == 70
):

    raise ValueError(
        "PHAFSTMN differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: PHAFSTMN matches CDC."
)


# ============================================================
# 19. INSULIN CONVERSION CONSISTENCY CHECK
# ============================================================

#
# CDC documentation states:
#
# LBDINSI = LBXIN * 6.0
#
# rounded to 2 decimal places.
#
# Verify the released values without changing them.
#

print("\n--- INSULIN CONVERSION CHECK ---")


mask = (
    df["LBXIN"].notna()
    &
    df["LBDINSI"].notna()
)


expected_insulin_si = (
    df.loc[
        mask,
        "LBXIN"
    ]
    * 6.0
).round(2)


actual_insulin_si = (
    df.loc[
        mask,
        "LBDINSI"
    ]
)


if not np.allclose(
    expected_insulin_si,
    actual_insulin_si,
    rtol=0,
    atol=1e-12
):

    raise ValueError(
        "LBDINSI conversion consistency check failed."
    )


print(
    "PASS: LBDINSI is consistent with "
    "LBXIN x 6.0."
)


# ============================================================
# 20. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Expected changes from original pandas XPT:
#
# WTSAF2YR:
#     396 tiny values -> 0
#
# PHAFSTHR:
#     57 tiny values -> 0
#
# PHAFSTMN:
#     70 tiny values -> 0
#
# NOTHING ELSE may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if col in DECIMAL_COLUMNS:

        current_values = (
            df[col]
            .to_numpy(
                dtype=float
            )
        )

    else:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 8 variables match expected "
    "corrected XPT values exactly."
)


# ============================================================
# 21. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO float_format.
# NO rounding.
#
# Decimal variables remain decimal:
#
# WTSAF2YR
# LBDGLUSI
# LBXIN
# LBDINSI
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 22. RE-READ EXPORTED CSV
# ============================================================

#
# float_precision="round_trip" is important
# for exact decimal-value fidelity checks.
#

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 23. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 8 CSV columns match "
    "expected corrected XPT values exactly."
)


# ============================================================
# 24. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 25. CHECK CSV FOR REMAINING TINY VALUES
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 26. WHOLE-NUMBER .0 FORMAT CHECK
# ============================================================

#
# IMPORTANT:
#
# DO NOT search the whole CSV for ".0".
#
# Decimal variables may legitimately contain:
#
#     5.0
#     10.0
#     0.0
#
# We check ONLY:
#
# SEQN
# LBXGLU
# PHAFSTHR
# PHAFSTMN
#

print("\n--- WHOLE-NUMBER FORMAT CHECK ---")


dot_zero_errors = []


with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(
        f
    )


    for row_number, row in enumerate(
        reader,
        start=2
    ):

        for col in INTEGER_COLUMNS:

            value = row[col]


            if (
                value != ""
                and
                value.endswith(".0")
            ):

                dot_zero_errors.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


                if len(
                    dot_zero_errors
                ) >= 20:

                    break


        if len(
            dot_zero_errors
        ) >= 20:

            break


print(
    "Unnecessary '.0' errors:",
    len(
        dot_zero_errors
    )
)


if dot_zero_errors:

    print(
        dot_zero_errors
    )

    raise ValueError(
        "Unexpected .0 formatting remains "
        "in whole-number fields."
    )


print(
    "PASS: SEQN and whole-number fields "
    "have no unnecessary .0."
)


# ============================================================
# 27. EXACT DECIMAL FIDELITY CHECK
# ============================================================

#
# Explicitly verify every genuine decimal variable.
#
# They must survive XPT -> correction -> CSV
# without numeric alteration.
#

print("\n--- DECIMAL VALUE FIDELITY ---")


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_failures.append(
            col
        )


if decimal_failures:

    raise ValueError(
        "Decimal values changed in:\n"
        + str(decimal_failures)
    )


print(
    "PASS: All genuine decimal variables "
    "preserved exactly."
)


# ============================================================
# 28. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print("\n--- SCIENTIFIC NOTATION CHECK ---")


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 29. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51641
    or
    csv_df["SEQN"].max()
    != 62158
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 30. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "GLU_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Decimal variables preserved:",
    ", ".join(
        DECIMAL_COLUMNS
    )
)


print(
    "Whole-number variables:",
    ", ".join(
        INTEGER_COLUMNS
    )
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0 in whole-number fields: 0 expected"
)


print(
    "Lab decimal values changed: 0 expected"
)

NHANES 2009-2010 GLU_F XPT -> CSV
Rows: 3,581
Columns: 8
PASS: Structure = 3,581 rows x 8 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 523

Tiny values by variable:
WTSAF2YR    396
PHAFSTMN     70
PHAFSTHR     57

PASS: Tiny-value pattern matches validated GLU_F pattern.

--- ZERO RESTORATION ---
WTSAF2YR: 396 restored to 0
PHAFSTHR: 57 restored to 0
PHAFSTMN: 70 restored to 0

PASS: 523 validated zero values restored.

--- GENUINE DECIMAL CHECK ---
WTSAF2YR: 3,181 genuine fractional values
LBDGLUSI: 3,386 genuine fractional values
LBXIN: 3,293 genuine fractional values
LBDINSI: 3,261 genuine fractional values

PASS: Genuine decimal variables identified:
    WTSAF2YR
    LBDGLUSI
    LBXIN
    LBDINSI

PASS: Genuine decimals will NOT be rounded or converted to Int64.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51641
Maximum: 62158
PASS: SEQN validated.

PASS: Whole-number fields formatted without unnecessary .0.
PASS: Lab decimal variab